In [1]:
from datetime import date
import hisepy
import os
import pandas as pd
import polars as pl
import re

In [2]:
if not os.path.isdir('output'):
    os.mkdir('output')

In [3]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

### Sample Metadata

In [4]:
meta_uuid = 'af25e3e7-25c1-4476-afb4-926bd201db8f'

In [5]:
meta_file = hisepy.cache_files([meta_uuid])[0]

In [6]:
meta = pl.read_csv(meta_file)

In [7]:
meta.shape

(868, 19)

In [8]:
meta.head()

,cohort.cohortGuid,subject.subjectGuid,subject.biologicalSex,subject.cmv,subject.bmi,subject.race,subject.ethnicity,subject.birthYear,subject.ageAtFirstDraw,subject.covidVaxDose1.daysSinceFirstVisit,subject.covidVaxDose2.daysSinceFirstVisit,sample.sampleKitGuid,sample.visitName,sample.drawDate,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit,specimen.specimenGuid,pipeline.fileGuid
i64,str,str,str,str,f64,str,str,i64,i64,f64,f64,str,str,str,i64,i64,str,str
0,"""BR1""","""BR1001""","""Female""","""Negative""",23.0,"""Caucasian""","""Non-Hispanic origin""",1987,32,null,null,"""KT00001""","""Flu Year 1 Day 0""","""2019-10""",32,0,"""PB00001-01""","""fec489f9-9a74-4635-aa91-d2bf09…"
1,"""BR1""","""BR1002""","""Male""","""Negative""",22.0,"""Caucasian""","""Non-Hispanic origin""",1991,28,440.0,461.0,"""KT00002""","""Flu Year 1 Day 0""","""2019-10""",28,0,"""PB00002-01""","""7c0c7979-eebd-4aba-b5b2-6e76b4…"
2,"""BR1""","""BR1003""","""Female""","""Negative""",21.0,"""Caucasian""","""Non-Hispanic origin""",1989,30,440.0,461.0,"""KT00003""","""Flu Year 1 Day 0""","""2019-10""",30,0,"""PB00003-01""","""40efd03a-cb2f-4677-af42-a056cb…"
3,"""BR1""","""BR1004""","""Male""","""Negative""",22.0,"""Caucasian""","""Non-Hispanic origin""",1989,30,543.0,563.0,"""KT00004""","""Flu Year 1 Day 0""","""2019-10""",30,0,"""PB00004-01""","""68fbcd34-1d63-461d-8195-df5b8d…"
4,"""BR1""","""BR1005""","""Female""","""Negative""",20.0,"""Caucasian""","""Non-Hispanic origin""",1992,27,451.0,492.0,"""KT00006""","""Flu Year 1 Day 0""","""2019-10""",27,0,"""PB00006-01""","""ea8d98e9-e99e-4dc6-9e78-9866e0…"


In [9]:
age_groups = {
    'BR1': 'Young Adult',
    'BR2': 'Older Adult'
}

In [10]:
meta = meta.with_columns(
    pl.Series(
        name = 'subject.ageGroup',
        values = [age_groups[c] for c in meta['cohort.cohortGuid']]
    )
)

In [11]:
meta.columns

['',
 'cohort.cohortGuid',
 'subject.subjectGuid',
 'subject.biologicalSex',
 'subject.cmv',
 'subject.bmi',
 'subject.race',
 'subject.ethnicity',
 'subject.birthYear',
 'subject.ageAtFirstDraw',
 'subject.covidVaxDose1.daysSinceFirstVisit',
 'subject.covidVaxDose2.daysSinceFirstVisit',
 'sample.sampleKitGuid',
 'sample.visitName',
 'sample.drawDate',
 'sample.subjectAgeAtDraw',
 'sample.daysSinceFirstVisit',
 'specimen.specimenGuid',
 'pipeline.fileGuid',
 'subject.ageGroup']

In [12]:
keep_meta_cols = [
    'cohort.cohortGuid',
    'subject.subjectGuid',
    'sample.sampleKitGuid',
    'subject.biologicalSex',
    'subject.birthYear',
    'subject.ageAtFirstDraw',
    'subject.ageGroup',
    'subject.race',
    'subject.ethnicity',
    'subject.cmv',
    'sample.visitName',
    'sample.drawDate',
    'sample.subjectAgeAtDraw',
    'sample.daysSinceFirstVisit'
]

In [13]:
meta = meta.select(keep_meta_cols)

In [14]:
meta.head()

cohort.cohortGuid,subject.subjectGuid,sample.sampleKitGuid,subject.biologicalSex,subject.birthYear,subject.ageAtFirstDraw,subject.ageGroup,subject.race,subject.ethnicity,subject.cmv,sample.visitName,sample.drawDate,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit
str,str,str,str,i64,i64,str,str,str,str,str,str,i64,i64
"""BR1""","""BR1001""","""KT00001""","""Female""",1987,32,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019-10""",32,0
"""BR1""","""BR1002""","""KT00002""","""Male""",1991,28,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019-10""",28,0
"""BR1""","""BR1003""","""KT00003""","""Female""",1989,30,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019-10""",30,0
"""BR1""","""BR1004""","""KT00004""","""Male""",1989,30,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019-10""",30,0
"""BR1""","""BR1005""","""KT00006""","""Female""",1992,27,"""Young Adult""","""Caucasian""","""Non-Hispanic origin""","""Negative""","""Flu Year 1 Day 0""","""2019-10""",27,0


In [15]:
data_uuids = [
    '7c2d668a-b851-4087-ba31-408de4ce889c',
    '00753770-4803-4947-a290-e7469c410067',
    '1a528317-bbf3-45b7-82c9-529577cf0b15',
    '55d5d20a-506a-459e-8315-7d66276fb8f9',
    '59f5f656-085f-4d0b-a240-68a9eb15e68e',
    'b494cce8-1314-4f6e-9666-42f0c6e1c702',
    'a2e01b9b-2ed9-4b3a-a43a-48a4bc2af007'
]

In [16]:
data_files = []
for uuid in data_uuids:
    data_files.append(hisepy.cache_files([uuid])[0])

In [17]:
data_list = []
for file in data_files:
    df = pd.read_csv(file)
    data_list.append(pl.DataFrame(df))

/tmp/ipykernel_1555/3137820088.py:3: DtypeWarning: Columns (53) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/tmp/ipykernel_1555/3137820088.py:3: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/tmp/ipykernel_1555/3137820088.py:3: DtypeWarning: Columns (32) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/tmp/ipykernel_1555/3137820088.py:3: DtypeWarning: Columns (27,28,29,32,33) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)
/tmp/ipykernel_1555/3137820088.py:3: DtypeWarning: Columns (34,35) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


### Update the first dataset
This is the bridging reference sample, so it's missing columns related to bridging

In [18]:
data_list[0] = data_list[0].with_columns(
    pl.col('LOD').alias('LOD_bridged'),
    pl.col('NPX').alias('NPX_bridged'),
    pl.lit(0).cast(pl.Float64).alias('BatchOffset')
)

In [19]:
rename_cols = {
    'NPX': 'olink.NPX_raw',
    'NPX_qry': 'olink.NPX_raw',
    'NPX_bridged': 'olink.NPX_norm',
    'PlateID': 'olink.plate_id',
    'PlateID_qry': 'olink.plate_id',
    'LOD': 'olink.LOD_raw',
    'LOD_qry': 'olink.LOD_raw',
    'LOD_bridged': 'olink.LOD_norm',
    'Assay': 'olink.assay',
    'SampleID': 'specimen.specimenGuid',
    'OlinkID': 'olink.assay_id',
    'Panel': 'olink.panel',
    'UniProt': 'olink.uniprot_id',
    'BatchOffset': 'olink.norm_offset'
}

In [20]:
renamed_data = []
for data in data_list:
    for old, new in rename_cols.items():
        if old in data.columns:
            data = data.rename({old: new})
    renamed_data.append(data)

In [21]:
select_cols = list(set(rename_cols.values()))

In [22]:
select_cols

['olink.assay_id',
 'olink.NPX_raw',
 'olink.panel',
 'olink.plate_id',
 'specimen.specimenGuid',
 'olink.uniprot_id',
 'olink.assay',
 'olink.LOD_raw',
 'olink.NPX_norm',
 'olink.LOD_norm',
 'olink.norm_offset']

In [23]:
selected_list = []
for data in renamed_data:
    data = data[select_cols]
    selected_list.append(data)
all_data = pl.concat(selected_list)

In [24]:
specimens = all_data['specimen.specimenGuid'].unique().to_list()
kits = [re.sub('PL([0-9]+)-.+','KT\\1',specimen) for specimen in specimens]
specimen_to_kit = dict(zip(specimens, kits))

In [25]:
all_data = all_data.with_columns(
    pl.Series(
        name = "sample.sampleKitGuid", 
        values = [specimen_to_kit[s] for s in all_data['specimen.specimenGuid']]
    ),
    pl.col('olink.plate_id').str.replace('_.+', '').alias('olink.batch_id')
)

In [26]:
all_data.shape

(3743263, 13)

In [27]:
all_data.head()

olink.assay_id,olink.NPX_raw,olink.panel,olink.plate_id,specimen.specimenGuid,olink.uniprot_id,olink.assay,olink.LOD_raw,olink.NPX_norm,olink.LOD_norm,olink.norm_offset,sample.sampleKitGuid,olink.batch_id
str,f64,str,str,str,str,str,f64,f64,f64,f64,str,str
"""OID21145""",0.3975,"""Neurology""","""20201752_SS200049_NEU_ONC""","""PL00205-01""","""P20333""","""TNFRSF1B""",-6.3078,0.3975,-6.3078,0.0,"""KT00205""","""20201752"""
"""OID21153""",0.164,"""Neurology""","""20201752_SS200049_NEU_ONC""","""PL00205-01""","""Q6UXH9""","""PAMR1""",-5.2385,0.164,-5.2385,0.0,"""KT00205""","""20201752"""
"""OID21136""",0.8896,"""Neurology""","""20201752_SS200049_NEU_ONC""","""PL00205-01""","""P16871""","""IL7R""",-5.0161,0.8896,-5.0161,0.0,"""KT00205""","""20201752"""
"""OID21119""",null,"""Neurology""","""20201752_SS200049_NEU_ONC""","""PL00205-01""","""Q8N6Q3""","""CD177""",null,null,null,0.0,"""KT00205""","""20201752"""
"""OID20867""",1.0911,"""Neurology""","""20201752_SS200049_NEU_ONC""","""PL00205-01""","""Q9NRG1""","""PRTFDC1""",1.1751,1.0911,1.1751,0.0,"""KT00205""","""20201752"""


In [28]:
all_data['olink.batch_id'].unique().sort()

olink.batch_id
str
"""20201752"""
"""20211036"""
"""20211037"""
"""20212223"""
"""Q-02017"""
"""Q-04064"""
"""Q-09225"""


In [29]:
test = all_data['specimen.specimenGuid'].str.contains('02284')

In [30]:
sum(test)

2944

In [31]:
order_cols = [
    'sample.sampleKitGuid',
    'specimen.specimenGuid',
    'olink.batch_id',
    'olink.plate_id',
    'olink.panel',
    'olink.assay',
    'olink.assay_id',
    'olink.uniprot_id',
    'olink.norm_offset',
    'olink.NPX_raw',
    'olink.NPX_norm',
    'olink.LOD_raw',
    'olink.LOD_norm'
]

In [32]:
soundlife_data = all_data.select(order_cols).filter(
    pl.col('sample.sampleKitGuid').is_in(meta['sample.sampleKitGuid'])
)

In [33]:
soundlife_data.shape

(1274064, 13)

In [34]:
len(soundlife_data['sample.sampleKitGuid'].unique())

867

In [35]:
len(soundlife_data['specimen.specimenGuid'].unique())

867

In [36]:
len(set(meta['sample.sampleKitGuid']).intersection(set(soundlife_data['sample.sampleKitGuid'])))

867

In [37]:
missing_data = meta.filter(
    ~pl.col('sample.sampleKitGuid').is_in(soundlife_data['sample.sampleKitGuid'])
)

In [38]:
missing_data

cohort.cohortGuid,subject.subjectGuid,sample.sampleKitGuid,subject.biologicalSex,subject.birthYear,subject.ageAtFirstDraw,subject.ageGroup,subject.race,subject.ethnicity,subject.cmv,sample.visitName,sample.drawDate,sample.subjectAgeAtDraw,sample.daysSinceFirstVisit
str,str,str,str,i64,i64,str,str,str,str,str,str,i64,i64
"""BR1""","""BR1037""","""KT02352""","""Female""",1984,36,"""Young Adult""","""Asian""","""Non-Hispanic origin""","""Positive""","""Flu Year 2 Day 7""","""2021-09""",37,564


In [39]:
soundlife_data = soundlife_data.join(
    meta,
    how = 'left',
    on = 'sample.sampleKitGuid'
)

In [40]:
out_file = 'output/sound_life_all_olink_{d}.csv'.format(d = date.today())

In [41]:
soundlife_data.write_csv(out_file)

## Upload data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [42]:
study_space_uuid = 'de025812-5e73-4b3c-9c3b-6d0eac412f2a'
title = 'Sound Life Olink Data for File Sets {d}'.format(d = date.today())

In [43]:
search_id = element_id()
search_id

'osmium-praseodymium-oxygen'

In [44]:
in_files = [meta_uuid] + data_uuids
in_files

['af25e3e7-25c1-4476-afb4-926bd201db8f',
 '7c2d668a-b851-4087-ba31-408de4ce889c',
 '00753770-4803-4947-a290-e7469c410067',
 '1a528317-bbf3-45b7-82c9-529577cf0b15',
 '55d5d20a-506a-459e-8315-7d66276fb8f9',
 '59f5f656-085f-4d0b-a240-68a9eb15e68e',
 'b494cce8-1314-4f6e-9666-42f0c6e1c702',
 'a2e01b9b-2ed9-4b3a-a43a-48a4bc2af007']

In [45]:
out_files = [out_file]
out_files

['output/sound_life_all_olink_2025-04-25.csv']

In [46]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

checking if conda environment can compile...
Cannot determine the current notebook.
1) /home/workspace/sound-life-scrna-analysis/olink-proteomics/Python_olink_data_fileset.ipynb
2) /home/workspace/sound-life-scrna-analysis/templates-and-examples/TE-R_template_h5ad_data_per_sample.ipynb
3) /home/workspace/sound-life-scrna-analysis/templates-and-examples/TE-R_template_generic_certpro.ipynb
Please select (1-3) 


 1


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': '4d26d6d5-b274-4c9b-b482-a56be3ecedfa',
 'ProcessId': '4f6c17f5-5a7d-4c2d-a3e7-a5dbe080d54c',
 'WorkflowId': '3d270835-4038-4171-a3e6-f9b6966cfc7b',
 'FileIds': ['c5f2c5e9-73a1-4e67-a9f0-4edb9113e27f']}

In [47]:
import session_info
session_info.show()